<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/13-foundation-models-multimodal-learning.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **基础模型与多模态学习** {#foundation-models-multimodal-learning}

基础模型（foundation model）在广泛数据上进行预训练，并被设计成可以通过 prompting、retrieval、adapter、fine-tuning 或与其他系统组合来支持多种下游用途。它的重要性来自**复用与杠杆效应**：一个上游表示、数据决策、接口或缺陷，可能影响许多不同部署。多模态学习进一步把这种复用扩展到文本、图像、音频、视频、深度、传感器流和结构化记录。

这个术语并不等于“任何大型神经网络”。[Stanford CRFM 报告](https://arxiv.org/abs/2108.07258)强调广泛训练和适配，同时把 foundation model 描述为尚不完整的 sociotechnical system。一个没有数据文档、适配契约、评估和监控的 checkpoint，并不是完整的基础模型项目。

![从广泛数据到多个适配部署以及反馈环路的基础模型生命周期。](assets/dl13-foundation-lifecycle.svg){fig-align="center" width="76%" fig-alt="经过治理的广泛数据流向预训练、适配与多个部署，监控反馈再返回训练系统。"}

*根据 [CRFM foundation-model 报告](https://arxiv.org/abs/2108.07258)讨论的生命周期与下游杠杆效应绘制的原创综合图。*

本章使用 scikit-learn 收录的 [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)，DOI 为 [10.24432/C50P49](https://doi.org/10.24432/C50P49)，采用 CC BY 4.0 许可。每张真实 8×8 图像会配上**人工构造且完全公开规则**的英文类别描述，例如 `a handwritten digit three`，以及 `odd, low` 这样的两个粗粒度属性。这些 caption 不是 UCI 原始 annotation。它们构成一个规模小、可审计的 image-text 机制实验；本章不会把结果表述成 VLM benchmark。

![真实数字图像与实验所用受控文本和属性模态的配对。](assets/dl13-controlled-pairs.svg){fig-align="center" width="76%" fig-alt="十张手写数字图像分别配有英文数字名称以及粗粒度奇偶性和大小属性。"}

*原创数据可视化。图像来自 UCI 衍生数据集；文本和属性由本章构造，并按照任务定义在每个 split 中都可获得。*

<details>
<summary><strong>PyTorch：建立共享 image-text 数据集与 provenance 契约</strong></summary>

```python
import copy
import hashlib
import math
import random
from collections import Counter

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1313):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1313, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1313,
    stratify=digits.target[holdout_idx],
)
train_x, train_y = all_images[train_idx], all_labels[train_idx]
val_x, val_y = all_images[val_idx], all_labels[val_idx]
test_x, test_y = all_images[test_idx], all_labels[test_idx]

digit_names = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
caption_templates = [
    "a handwritten digit {}",
    "an image of the number {}",
    "the optical character {}",
]
all_template_texts = [template.format(name) for name in digit_names for template in caption_templates]
vocabulary = {"<pad>": 0, "<unk>": 1}
for token in sorted({token for text in all_template_texts for token in text.split()}):
    vocabulary[token] = len(vocabulary)


def encode_texts(texts, max_length=7):
    ids = torch.zeros(len(texts), max_length, dtype=torch.long)
    mask = torch.zeros(len(texts), max_length, dtype=torch.bool)
    for row, text in enumerate(texts):
        tokens = [vocabulary.get(token, vocabulary["<unk>"]) for token in text.split()][:max_length]
        ids[row, :len(tokens)] = torch.tensor(tokens)
        mask[row, :len(tokens)] = True
    return ids, mask


def captions_for(indices, labels):
    return [caption_templates[int(index) % len(caption_templates)].format(digit_names[int(label)])
            for index, label in zip(indices, labels)]


train_text_ids, train_text_mask = encode_texts(captions_for(train_idx, train_y))
val_text_ids, val_text_mask = encode_texts(captions_for(val_idx, val_y))
test_text_ids, test_text_mask = encode_texts(captions_for(test_idx, test_y))


def hint_features(labels):
    # These are intentionally supplied side information, not hidden targets.
    return torch.stack([(labels % 2 == 0).float(), (labels >= 5).float()], dim=1)


train_hints, val_hints, test_hints = hint_features(train_y), hint_features(val_y), hint_features(test_y)


class ImageEncoder(nn.Module):
    def __init__(self, output_dim=32, hidden_dim=96):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(64, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, images):
        return self.network(images)


@torch.no_grad()
def classifier_accuracy(model, images, labels):
    model.eval()
    output = model(images)
    logits = output[0] if isinstance(output, tuple) else output
    return float((logits.argmax(1) == labels).float().mean())


assert all_images.shape == (1797, 1, 8, 8)
assert len(set(train_idx) & set(test_idx)) == 0
assert train_text_ids.shape == (1257, 7)
assert vocabulary["<pad>"] == 0
print({
    "split": (len(train_idx), len(val_idx), len(test_idx)),
    "vocabulary size": len(vocabulary),
    "example pair": (captions_for(train_idx[:1], train_y[:1])[0], int(train_y[0])),
    "constructed modalities": ["image", "class description", "parity/magnitude hint"],
})
```

</details>

划分单位是原始图像索引。文本只在划分后生成，因此同一图像的不同模板不会跨越训练集与测试集泄漏。粗粒度 hint 在 fusion 实验中被视为可用输入模态；它们不会被描述成自然采集的 metadata。

### **什么是基础模型** {#what-is-a-foundation-model}

基础模型模式通常由三个属性区分：

1. **广泛预训练：**上游目标覆盖足够多的数据、任务或模态，从而学习可复用结构。
2. **可适配性：**用户能够通过 prompt、retrieval、head、PEFT、fine-tuning、tool 或 structured output 指定新行为。
3. **下游杠杆效应：**多个系统继承相同表示，也继承相同能力、bias 与 vulnerability。

规模通常是重要的推动因素，但广度与接口比某个参数门槛更重要。一个跨多家医院预训练、可以适配多个临床任务的小型领域模型也可能符合这种模式；一个只为固定标签集训练的巨大分类器则未必符合。“通用”也不等于“普遍胜任”。能力取决于训练支持范围、输入接口、上下文、评估和部署环境。

一个有用的抽象是

$$
z=F_{\theta}(x, c),\qquad
\hat y=A_{\phi}(z, q),
$$

其中 $F_{\theta}$ 是广泛预训练的 base，$x$ 是观测数据，$c$ 是上下文或另一模态，$A_{\phi}$ 是适配/接口层，$q$ 指定下游请求。这种分离是概念性的而非强制性的：in-context learning 可以保持 $\theta$ 不变，完整 fine-tuning 会修改它，而 retrieval 在推理时改变 $c$。

相同的杠杆效应也会制造相关风险。一个 representation shortcut 可能传播到每个适配分类器；记忆的隐私数据可能通过多个接口暴露；tokenizer 或图像预处理缺陷可能静默影响看似无关的应用。因此，基础模型评估必须同时包含**上游属性**和**特定下游用途证据**；一个平均 benchmark 无法认证所有 adaptation。

### **缩放定律与计算最优训练** {#scaling-laws-compute-optimal-training}

缩放定律（scaling laws）是连接 loss、模型规模 $N$、数据暴露量 $D$ 与训练计算量 $C$ 的经验规律。常见的可分形式为

$$
L(N,D)\approx L_{\infty}+aN^{-\alpha}+bD^{-\beta},
$$

其中 $L_{\infty}$ 是所选分布和目标下不可约的下限。常数与指数只能在特定 regime 中拟合，并不是普适物理定律。[Kaplan 等人](https://arxiv.org/abs/2001.08361)记录了语言模型 loss 的 power-law behavior。[Hoffmann 等人](https://arxiv.org/abs/2203.15556)随后使用 IsoFLOP 分析，并在其设定下发现 compute-optimal model size 与 token count 应近似同步增长。

对于 dense Transformer 训练，粗略数量级估计为 $C\approx 6ND$ FLOPs。在固定 $C$ 下，增大 $N$ 会使 token update 数量下降到 $D\approx C/(6N)$。因此模型可能处于**训练不足（undertrained）**状态：购买了参数容量，却没有让它看到足够数据。部署成本也必须考虑；预训练计算量相近的两个模型，可能具有完全不同的推理内存和延迟。

![固定计算预算在模型规模与数据暴露量之间的概念性分配。](assets/dl13-scaling-laws.svg){fig-align="center" width="74%" fig-alt="概念性 IsoFLOP 图展示固定计算量下，模型参数量与训练 token 数之间的平衡 frontier。"}

*根据 [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556) 中的 IsoFLOP 推理绘制的原创概念图，不复刻论文拟合系数。*

下面的小型实验改变 hidden width 和每类样本数，同时固定目标与 optimizer。它规模太小，不能估计普遍缩放定律；其用途是展示在拟合之前应建立的测量表。

<details>
<summary><strong>PyTorch：建立小型模型-数据 scaling table</strong></summary>

```python
class WidthClassifier(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(64, width), nn.ReLU(), nn.Linear(width, 10)
        )

    def forward(self, images):
        return self.network(images)


def balanced_subset(labels, per_class, seed):
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in range(10):
        candidates = torch.where(labels == label)[0]
        chosen.append(candidates[torch.randperm(len(candidates), generator=generator)[:per_class]])
    return torch.cat(chosen)


def fit_scaling_point(width, per_class, epochs=35):
    seed_everything(1300 + width + per_class)
    subset = balanced_subset(train_y, per_class, seed=1300 + per_class)
    model = WidthClassifier(width)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
    loader = DataLoader(
        TensorDataset(train_x[subset], train_y[subset]), batch_size=64,
        shuffle=True, generator=torch.Generator().manual_seed(1300 + per_class),
    )
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        validation_loss = float(F.cross_entropy(model(val_x), val_y))
        validation_accuracy = classifier_accuracy(model, val_x, val_y)
    parameters = sum(p.numel() for p in model.parameters())
    compute_proxy = parameters * len(subset) * epochs
    return {"width": width, "examples": len(subset), "parameters": parameters,
            "compute_proxy": compute_proxy, "val_loss": validation_loss,
            "val_accuracy": validation_accuracy}


scaling_rows = [fit_scaling_point(width, per_class)
                for width in (16, 32, 64) for per_class in (10, 40)]
for row in scaling_rows:
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in row.items()})

assert len(scaling_rows) == 6
assert all(math.isfinite(row["val_loss"]) for row in scaling_rows)
assert len({row["compute_proxy"] for row in scaling_rows}) == 6
```

</details>

可信 scaling study 会在相同计算预算上训练多个点，公平调整 learning schedule，重复随机种子，并检验 held-out prediction。Accuracy 容易饱和且不连续，通常不适合拟合；pretraining cross-entropy 更有信息。超出观测尺度的外推必须报告不确定性，而且 scaling curve 本身并不能说明安全性、数据权利、事实性或下游效用。

### **训练数据整理与治理** {#training-data-curation-governance}

在 foundation-model 规模上，数据就是算法的一部分。来源 mixture、语言与地理覆盖、过滤阈值、deduplication、时间截止点、annotation policy、license、个人信息和 opt-out 机制，共同决定模型能学习什么，以及哪些群体的错误被隐藏。[DataComp](https://arxiv.org/abs/2304.14108)展示了如何在标准化训练计算量下研究 dataset design，而不是把它当成没有文档的预处理步骤。

![从数据获取和文档记录到过滤、去重与审计的整理流程。](assets/dl13-data-curation.svg){fig-align="center" width="76%" fig-alt="五阶段数据整理流程包括 provenance、文档记录、过滤、去重、subgroup 和 contamination audit，以及反馈环路。"}

*参考 [DataComp](https://arxiv.org/abs/2304.14108) 与 [Datasheets for Datasets](https://arxiv.org/abs/1803.09010) 的文档原则绘制的原创综合图。*

Deduplication 有多个作用：避免流行或复制样本获得非预期权重，降低 train-test contamination，并防止评估结果过度乐观。Exact hashing 可以检测完全相同的 byte；near-duplicate detection 则需要 perceptual、semantic 或 document-level grouping。如果不同记录共享信息，split 应该按 duplicate group、source document、speaker、patient 或 video 进行，而不是按孤立 row 随机划分。

多模态 pair 还会引入 alignment quality 问题。流畅 caption 可能只描述图像很小一部分；alt text 可能是文件名或 SEO 文本；音频与视频可能存在时间偏移；synthetic caption 会放大 teacher error。过度过滤也可能清除 dialect、minority context、困难样本和安全相关数据。因此治理既要记录保留了什么，也要记录删除了什么。

<details>
<summary><strong>Python：审计重复图像与损坏的 image-text pair</strong></summary>

```python
def image_hash(image):
    return hashlib.sha256(image.numpy().tobytes()).hexdigest()


# Build a deliberately contaminated copy without changing the clean experiment source.
contaminated_images = torch.cat([train_x, train_x[:80]], dim=0)
contaminated_image_labels = torch.cat([train_y, train_y[:80]], dim=0)
contaminated_caption_labels = contaminated_image_labels.clone()
corrupt_rows = torch.arange(0, 120, 4)
contaminated_caption_labels[corrupt_rows] = (contaminated_caption_labels[corrupt_rows] + 3) % 10
hashes = [image_hash(image) for image in contaminated_images]
hash_counts = Counter(hashes)

seen, keep = set(), []
for row, digest in enumerate(hashes):
    pair_matches = contaminated_image_labels[row] == contaminated_caption_labels[row]
    if digest not in seen and bool(pair_matches):
        seen.add(digest)
        keep.append(row)

audit_report = {
    "rows before": len(contaminated_images),
    "duplicate rows": sum(count - 1 for count in hash_counts.values()),
    "caption mismatches": int((contaminated_image_labels != contaminated_caption_labels).sum()),
    "rows after exact-dedup and pair audit": len(keep),
}
assert audit_report["duplicate rows"] == 80
assert audit_report["caption mismatches"] == len(corrupt_rows)
assert len(set(hashes[row] for row in keep)) == len(keep)
print(audit_report)
```

</details>

这个审计能够检测损坏，是因为受控 caption 由已知标签生成。真实 web pair 很少有这种 ground truth，通常需要质量模型、人工抽样、来源规则与下游 ablation。过滤模型本身也应该版本化，并接受 subgroup bias 评估。数据治理在发布后仍会继续：删除请求、新发现的 contamination，以及变化的法律或伦理限制，都要求从 source record 追溯到 model version。

### **Dense 模型与混合专家模型** {#dense-models-mixture-of-experts}

Dense layer 对每个 token 使用相同参数。Mixture-of-Experts（MoE）层包含多个 expert network，但每个 token 只会被路由到 top $k$ 个 expert。对于 router probability $p_e(h)$ 和所选集合 $S(h)$，

$$
\operatorname{MoE}(h)=\sum_{e\in S(h)}p_e(h)E_e(h),\qquad |S(h)|=k\ll E.
$$

总参数容量随 expert 数量 $E$ 增长，而 active token-level computation 主要随 $k$ 增长。这是 conditional computation，不是免费的规模扩展。Expert weight 仍需存储和通信；token 可能超过 expert capacity；router 可能坍缩到少数 expert；分布式 all-to-all communication 可能支配运行时间。[Switch Transformer](https://arxiv.org/abs/2101.03961)把 routing 简化为 top-1 expert，同时强调了效率与稳定性挑战。

![Top-k routing 把每个 token 发送到部分 expert，再组合其输出。](assets/dl13-moe-routing.svg){fig-align="center" width="76%" fig-alt="Token state 经过 router 进入被选 expert 并组合输出，未被选择的 expert 被跳过。"}

*根据 [Switch Transformer](https://arxiv.org/abs/2101.03961) 的 sparse-routing 机制绘制的原创教学图。*

常见 load-balancing auxiliary objective 会鼓励平均 router probability 与实际 token fraction 在 expert 间保持一致，但这并不保证 semantic specialization。下面的小型 top-1 模型同时报告总参数量和近似 active expert 参数量。

<details>
<summary><strong>PyTorch：比较 dense 与 sparse expert 分类器</strong></summary>

```python
class DenseDigitModel(nn.Module):
    def __init__(self, hidden=48):
        super().__init__()
        self.input = nn.Linear(64, hidden)
        self.block = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.head = nn.Linear(hidden, 10)

    def forward(self, images, return_aux=False):
        hidden = F.relu(self.input(images.flatten(1)))
        hidden = hidden + self.block(hidden)
        logits = self.head(F.relu(hidden))
        return (logits, logits.new_zeros(()), None) if return_aux else logits


class SparseMoEDigitModel(nn.Module):
    def __init__(self, hidden=48, experts=4):
        super().__init__()
        self.input = nn.Linear(64, hidden)
        self.router = nn.Linear(hidden, experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            for _ in range(experts)
        ])
        self.head = nn.Linear(hidden, 10)

    def forward(self, images, return_aux=False):
        hidden = F.relu(self.input(images.flatten(1)))
        probabilities = self.router(hidden).softmax(dim=-1)
        routes = probabilities.argmax(dim=-1)
        expert_output = torch.zeros_like(hidden)
        fractions = []
        for expert_index, expert in enumerate(self.experts):
            selected = routes == expert_index
            fractions.append(selected.float().mean())
            if selected.any():
                expert_output[selected] = expert(hidden[selected]) * probabilities[selected, expert_index].unsqueeze(1)
        fractions = torch.stack(fractions)
        mean_probability = probabilities.mean(dim=0)
        assignment_balance = len(self.experts) * torch.sum(fractions.detach() * mean_probability)
        probability_balance = len(self.experts) * torch.sum(mean_probability.square())
        balance_loss = assignment_balance + probability_balance
        logits = self.head(F.relu(hidden + expert_output))
        return (logits, balance_loss, fractions) if return_aux else logits


def train_routed_model(model, epochs=45):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits, auxiliary, _ = model(train_x, return_aux=True)
        (F.cross_entropy(logits, train_y) + 0.03 * auxiliary).backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        logits, _, fractions = model(test_x, return_aux=True)
    return float((logits.argmax(1) == test_y).float().mean()), fractions


seed_everything(1330)
dense_model, moe_model = DenseDigitModel(), SparseMoEDigitModel()
dense_accuracy, _ = train_routed_model(dense_model)
moe_accuracy, expert_fractions = train_routed_model(moe_model)
dense_parameters = sum(p.numel() for p in dense_model.parameters())
moe_parameters = sum(p.numel() for p in moe_model.parameters())
one_expert_parameters = sum(p.numel() for p in moe_model.experts[0].parameters())
active_proxy = moe_parameters - sum(p.numel() for p in moe_model.experts.parameters()) + one_expert_parameters

assert abs(float(expert_fractions.sum()) - 1.0) < 1e-5
print({
    "dense accuracy/parameters": (round(dense_accuracy, 3), dense_parameters),
    "MoE accuracy/total parameters": (round(moe_accuracy, 3), moe_parameters),
    "MoE approximate active parameters": active_proxy,
    "expert token fractions": [round(float(value), 3) for value in expert_fractions],
})
```

</details>

这个 toy router 处理整张图像而不是 Transformer token，`active_proxy` 也忽略了通信与 router 成本。它演示的是核算方法，不是 production speed。在真实 MoE 训练中，应报告 expert utilization、dropped token、capacity factor、router entropy、all-to-all time、总参数与 active FLOPs。均衡 router 仍可能学到重复 expert；必须通过实验检验 specialization，不能从 expert ID 推断。

### **模态编码器与 Tokenization** {#modality-encoders-tokenization}

多模态系统必须把异构信号转化为计算单元。文本使用 subword 或 byte token；图像使用 pixel、patch、region 或学习到的 visual token；音频使用 waveform sample 或 time-frequency frame；视频使用 frame 或 space-time tubelet；结构化记录使用 typed field 与 missingness indicator。

Tokenization 同时是信息决策和成本决策。对于高度 $H$、宽度 $W$、patch size $P$ 的图像，non-overlapping patch encoder 会产生 $T_I=HW/P^2$ 个 token。含 $F$ 帧、temporal tubelet size 为 $P_t$ 的视频大约产生 $T_V=FHW/(P_tP^2)$ 个 token。完整 self-attention memory 按 $O(T^2)$ 增长，因此分辨率和持续时间既是数据参数，也是系统参数。

![图像、音频、视频与文本需要不同的采样和 token-budget 契约。](assets/dl13-modality-tokens.svg){fig-align="center" width="76%" fig-alt="图像被切成 patch，音频被切成时间 frame，视频被切成时空 tubelet，文本被切成 subword token。"}

*原创教学图。视觉组织参考 ViT 类模型、wav2vec 类音频模型与 [VideoMAE](https://arxiv.org/abs/2203.12602) 使用的模态 tokenizer。*

每种模态通常都会获得 position information 与 modality/type embedding。Padding mask 用于区分不存在的 token 与有效 silence 或黑色 pixel。时间对齐和采样率必须保留在 metadata 中，否则形状相同的两个 tensor 也可能表示不同物理时间区间。

<details>
<summary><strong>PyTorch：跟踪 image patch 与 text token 到共享宽度</strong></summary>

```python
def patchify(images, patch_size=2):
    patches = F.unfold(images, kernel_size=patch_size, stride=patch_size)
    return patches.transpose(1, 2)


shared_width = 32
image_patch_projection = nn.Linear(4, shared_width)
text_token_embedding = nn.Embedding(len(vocabulary), shared_width, padding_idx=0)
sample_patches = patchify(train_x[:5])
image_tokens = image_patch_projection(sample_patches)
text_tokens = text_token_embedding(train_text_ids[:5])

assert sample_patches.shape == (5, 16, 4)
assert image_tokens.shape == (5, 16, shared_width)
assert text_tokens.shape == (5, 7, shared_width)
assert train_text_mask[:5].shape == (5, 7)
print({
    "image token path": [(5, 1, 8, 8), tuple(sample_patches.shape), tuple(image_tokens.shape)],
    "text token path": [tuple(train_text_ids[:5].shape), tuple(text_tokens.shape)],
    "valid text tokens": train_text_mask[:5].sum(dim=1).tolist(),
})
```

</details>

相同宽度不等于相同语义。Encoder 可以完全独立，只在 projection head 处对齐；也可以让各模态进入统一 Transformer。独立 encoder 保留 specialist inductive bias，并更容易处理 missing modality。统一 token stream 允许丰富交互，却需要仔细处理 normalization、sampling 与 capacity allocation，避免 token 数量更多的模态支配模型。

### **Early、Late 与 Intermediate Fusion** {#early-late-intermediate-fusion}

Fusion 回答的是：**各模态在什么时候交换信息？**

- **Early fusion** 组合原始或轻度编码输入，可以建模低层交互，但需要对齐 sampling，并会使 preprocessing 高度耦合。
- **Intermediate fusion** 先由 specialist encoder 产生特征，再使用 concatenation、gating、cross-attention 或 connector，是 modularity 与 interaction 之间常见的折中。
- **Late fusion** 组合 prediction 或 score，便于模块化部署，也可以单独校准每个 expert，但无法学习细粒度 token-level correspondence。

![Early、intermediate 与 late fusion 的区别在于模态信息何时交互。](assets/dl13-fusion-taxonomy.svg){fig-align="center" width="76%" fig-alt="三个 panel 分别展示在编码前、通过中间 connector 或在独立决策后组合模态。"}

*综合标准多模态 fusion pattern 绘制的原创比较图。*

下面的受控任务使用图像和两个明确提供的 bit 预测数字：even/odd 以及 below/above five。这些属性会减少歧义，但不能唯一确定十个类别。这是刻意定义的任务，不是意外 label leakage；推理时同样能够访问这些属性。Missing-modality accuracy 通过把 hint 替换成零来测量。

<details>
<summary><strong>PyTorch：比较 early、intermediate 与 late fusion</strong></summary>

```python
class FusionClassifier(nn.Module):
    def __init__(self, mode, hidden=48):
        super().__init__()
        self.mode = mode
        if mode == "early":
            self.joint = nn.Sequential(nn.Linear(66, hidden), nn.ReLU(), nn.Linear(hidden, 10))
        else:
            self.image_branch = nn.Sequential(nn.Linear(64, hidden), nn.ReLU())
            self.hint_branch = nn.Sequential(nn.Linear(2, 16), nn.ReLU())
            if mode == "intermediate":
                self.head = nn.Linear(hidden + 16, 10)
            elif mode == "late":
                self.image_head = nn.Linear(hidden, 10)
                self.hint_head = nn.Linear(16, 10)
            else:
                raise ValueError(mode)

    def forward(self, images, hints):
        flat = images.flatten(1)
        if self.mode == "early":
            return self.joint(torch.cat([flat, hints], dim=1))
        image_hidden = self.image_branch(flat)
        hint_hidden = self.hint_branch(hints)
        if self.mode == "intermediate":
            return self.head(torch.cat([image_hidden, hint_hidden], dim=1))
        return 0.75 * self.image_head(image_hidden) + 0.25 * self.hint_head(hint_hidden)


def train_fusion(model, epochs=45):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = F.cross_entropy(model(train_x, train_hints), train_y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        full = float((model(test_x, test_hints).argmax(1) == test_y).float().mean())
        missing = float((model(test_x, torch.zeros_like(test_hints)).argmax(1) == test_y).float().mean())
    return full, missing


fusion_models, fusion_results = {}, {}
for offset, mode in enumerate(("early", "intermediate", "late")):
    seed_everything(1340 + offset)
    fusion_models[mode] = FusionClassifier(mode)
    fusion_results[mode] = train_fusion(fusion_models[mode])

assert set(fusion_results) == {"early", "intermediate", "late"}
print({mode: {"all modalities": round(scores[0], 3), "hint missing": round(scores[1], 3)}
       for mode, scores in fusion_results.items()})
```

</details>

Fusion 评估应该包含 unimodal baseline、modality dropout、冲突输入、timestamp misalignment 与 calibration。如果一个模态更容易预测训练目标，多模态模型可能完全忽略另一模态。Attention weight 或更大的 ablation drop 可以提示模型使用了某模态，却不能证明 causal grounding。任务中必须包含每个模态都不可替代的样本，才能真正检验 complementarity。

### **Cross-Attention 与共享 Embedding 空间** {#cross-attention-shared-embedding-spaces}

Shared embedding 和 cross-attention 解决相关但不同的问题。Dual encoder 把每种模态映射为一个 global vector，并常用 cosine similarity 计算兼容度：

$$
s(x,t)=\frac{f_I(x)^\top f_T(t)}{\tau},
$$

其中归一化的 image embedding 与 text embedding 共享维度，$\tau$ 是 temperature。Global vector 支持高效 retrieval，因为两侧可以独立建立索引；代价是压缩局部细节。

Cross-attention 保留 token-level state。Image query $Q_I$ 可以关注 text key/value $K_T,V_T$：

$$
\operatorname{CrossAttn}(Q_I,K_T,V_T)=
\operatorname{softmax}\left(\frac{Q_IK_T^\top}{\sqrt{d_k}}+M\right)V_T.
$$

$M$ 用于屏蔽 padding 或无效 alignment。Cross-attention 支持 grounding 与 conditional reasoning，却需要对每一对输入进行联合计算，因此大规模 retrieval 成本更高。

![Shared embedding alignment 与 cross-attention fusion 支持不同操作。](assets/dl13-alignment-cross-attention.svg){fig-align="center" width="76%" fig-alt="左侧对齐 global image/text vector 以进行 retrieval；右侧让 image patch query 关注 text token key/value 以进行局部交互。"}

*根据 [CLIP](https://proceedings.mlr.press/v139/radford21a.html) 的 dual-encoder 目标与 [Flamingo](https://arxiv.org/abs/2204.14198) 等模型使用的 cross-attention connector 绘制的原创综合图。*

由于许多训练图像共享相同类别描述，如果把所有 off-diagonal pair 都当作 negative，就会产生 false negative。下面的 loss 把数字标签相同的所有 pair 都视为 positive。

<details>
<summary><strong>PyTorch：训练 multi-positive image-text dual encoder 并跟踪 cross-attention</strong></summary>

```python
class DualEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.image_encoder = ImageEncoder(output_dim=embedding_dim)
        self.token_embedding = nn.Embedding(len(vocabulary), embedding_dim, padding_idx=0)
        self.text_projection = nn.Linear(embedding_dim, embedding_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.12)))

    def encode_image(self, images):
        return F.normalize(self.image_encoder(images), dim=-1)

    def encode_text(self, token_ids, token_mask):
        tokens = self.token_embedding(token_ids)
        mask = token_mask.unsqueeze(-1)
        pooled = (tokens * mask).sum(1) / mask.sum(1).clamp_min(1)
        return F.normalize(self.text_projection(pooled), dim=-1)

    def similarities(self, images, token_ids, token_mask):
        scale = self.logit_scale.exp().clamp(max=100)
        return scale * self.encode_image(images) @ self.encode_text(token_ids, token_mask).T


def multi_positive_loss(scores, labels):
    positives = labels[:, None] == labels[None, :]
    numerator_i = torch.logsumexp(scores.masked_fill(~positives, -1e9), dim=1)
    denominator_i = torch.logsumexp(scores, dim=1)
    numerator_t = torch.logsumexp(scores.T.masked_fill(~positives.T, -1e9), dim=1)
    denominator_t = torch.logsumexp(scores.T, dim=1)
    return 0.5 * ((denominator_i - numerator_i).mean() + (denominator_t - numerator_t).mean())


seed_everything(1350)
dual_encoder = DualEncoder()
optimizer = torch.optim.AdamW(dual_encoder.parameters(), lr=2e-3, weight_decay=1e-4)
loader = DataLoader(TensorDataset(train_x, train_text_ids, train_text_mask, train_y),
                    batch_size=160, shuffle=True, generator=torch.Generator().manual_seed(1350))
for _ in range(45):
    dual_encoder.train()
    for batch_images, batch_ids, batch_mask, batch_labels in loader:
        optimizer.zero_grad()
        scores = dual_encoder.similarities(batch_images, batch_ids, batch_mask)
        loss = multi_positive_loss(scores, batch_labels)
        loss.backward()
        optimizer.step()

# Trace a cross-attention connector using the trained text-token embedding.
patch_projection = nn.Linear(4, 32)
cross_attention = nn.MultiheadAttention(32, num_heads=4, batch_first=True)
query_tokens = patch_projection(patchify(test_x[:4]))
key_value_tokens = dual_encoder.token_embedding(test_text_ids[:4])
fused_tokens, attention_weights = cross_attention(
    query_tokens, key_value_tokens, key_value_tokens,
    key_padding_mask=~test_text_mask[:4], need_weights=True,
)

assert fused_tokens.shape == (4, 16, 32)
assert attention_weights.shape == (4, 16, 7)
assert torch.isfinite(loss)
print({"final contrastive loss": round(float(loss.detach()), 3),
       "cross-attention output": tuple(fused_tokens.shape),
       "attention map": tuple(attention_weights.shape)})
```

</details>

Multi-positive mask 使用标签，是因为本章文本只包含 class-level semantics。真实 image-caption corpus 可能在没有类别标签的情况下包含多个有效 caption，而且一些表面上的 negative 仍可能描述同一概念。因此，batch composition 本身就定义了 contrastive task。Temperature、distributed negative gathering、duplicate handling 与 caption specificity 都会改变学习到的几何结构。

### **视觉-语言模型** {#vision-language-models}

视觉-语言模型（vision-language models，VLMs）包含多类架构：

- **Dual encoder** 对齐独立 image/text embedding，用于 retrieval 与 zero-shot classification，例如 [CLIP](https://proceedings.mlr.press/v139/radford21a.html)。
- **Fusion encoder** 联合处理 visual/textual token，用于 matching、grounding 与判别式 question answering。
- **Vision-to-language generator** 把 visual encoder 连接到 language decoder，用于 captioning、VQA 与 instruction following。[Flamingo](https://arxiv.org/abs/2204.14198)使用 gated cross-attention 连接预训练 vision/language component；[BLIP](https://arxiv.org/abs/2201.12086)在处理 noisy caption 的同时组合 understanding 与 generation objective。

CLIP 类 zero-shot classification 把 class name 转换为 classifier weight。对于类别 prompt $t_c$ 和图像 $x$，预测为

$$
\hat y=\arg\max_c f_I(x)^\top f_T(t_c).
$$

因此分类器依赖语言措辞。Prompt ensembling 会对每类多个归一化 text embedding 取平均，减少对单一模板的敏感性，但不会产生预训练中不存在的知识。

<details>
<summary><strong>PyTorch：评估 zero-shot classification 与 prompt sensitivity</strong></summary>

```python
@torch.no_grad()
def text_prototypes(template):
    texts = [template.format(name) for name in digit_names]
    token_ids, token_mask = encode_texts(texts)
    return dual_encoder.encode_text(token_ids, token_mask)


@torch.no_grad()
def zero_shot_score(images, labels, prototypes):
    image_embeddings = dual_encoder.encode_image(images)
    predictions = (image_embeddings @ prototypes.T).argmax(1)
    return float((predictions == labels).float().mean())


dual_encoder.eval()
template_prototypes = [text_prototypes(template) for template in caption_templates]
template_scores = [zero_shot_score(test_x, test_y, prototypes) for prototypes in template_prototypes]
ensemble_prototypes = F.normalize(torch.stack(template_prototypes).mean(dim=0), dim=-1)
ensemble_score = zero_shot_score(test_x, test_y, ensemble_prototypes)

assert ensemble_prototypes.shape == (10, 32)
print({
    "per-template zero-shot accuracy": [round(score, 3) for score in template_scores],
    "prompt ensemble accuracy": round(ensemble_score, 3),
})
```

</details>

VLM 输出可能语言流畅，却缺少视觉支持。评估必须区分 object recognition、OCR、spatial reasoning、counting、temporal understanding 与 hallucination。Dataset overlap 与 prompt-format sensitivity 可能夸大表面 zero-shot ability。Grounding test 应在问题不变时替换图像、遮挡相关区域，并同时包含 answerable/unanswerable pair。流畅解释并不能证明模型使用了正确视觉区域。

### **音频、视频与多模态序列模型** {#audio-video-multimodal-sequence-models}

音频与视频引入了时间。Audio representation 必须保留 sampling rate、channel layout、window size 与 hop length；video representation 必须选择 frame rate、spatial resolution、clip duration 与 temporal tubelet size。同一个十秒事件可能表示成几十个 text token、几百个 audio frame 或几千个 video patch；简单拼接所有 token 会造成严重不平衡。

管理长多模态 stream 的架构策略包括：

- local 或 factorized spatial-temporal attention；
- temporal pooling、striding、learned resampling 或 latent bottleneck；
- modality-specific encoder 加 sparse cross-attention；
- 面向长录音的 memory、recurrence 或 retrieval；
- 高 mask ratio 的 masked prediction，例如 [VideoMAE](https://arxiv.org/abs/2203.12602)。

[ImageBind](https://arxiv.org/abs/2305.05665)把 image、text、audio、depth、thermal 与 IMU representation 对齐到一个空间，并使用 image-paired data 作为 binding signal。这说明并非每一对模态都需要直接监督；同时也会产生 hub-modality risk：在 image anchor 中表达不充分的概念可能对齐较弱。

<details>
<summary><strong>Python：在选择 attention 前比较 modality token budget</strong></summary>

```python
def modality_budget(image_hw=(224, 224), image_patch=16, audio_seconds=10,
                    audio_hop_ms=20, video_frames=32, tubelet=2):
    image_tokens = (image_hw[0] // image_patch) * (image_hw[1] // image_patch)
    audio_tokens = int(audio_seconds * 1000 / audio_hop_ms)
    video_tokens = (video_frames // tubelet) * image_tokens
    total_tokens = image_tokens + audio_tokens + video_tokens
    return {
        "image tokens": image_tokens,
        "audio frame tokens": audio_tokens,
        "video tubelet tokens": video_tokens,
        "concatenated tokens": total_tokens,
        "full-attention score entries": total_tokens ** 2,
    }


token_budget = modality_budget()
pooled_budget = modality_budget(image_hw=(112, 112), image_patch=16,
                                audio_seconds=10, audio_hop_ms=40,
                                video_frames=16, tubelet=4)
assert pooled_budget["full-attention score entries"] < token_budget["full-attention score entries"]
print({"high-resolution": token_budget, "pooled/strided": pooled_budget})
```

</details>

Token count 不等于 information content。激进 pooling 可能删除短声音、视频文字或瞬时动作；uniform frame sampling 可能错过稀有事件；voice activity detection 可能删除有意义的 silence；异步 sensor 需要 timestamp，而不是假设 index 对齐。评估应该按持续时间和事件频率分层，并测试 temporal shuffling、modality delay 与 missing segment。

### **多模态预训练目标** {#multimodal-pretraining-objectives}

不同 objective 教授不同关系：

- **Contrastive alignment** 拉近匹配的 global representation，适合 retrieval。
- **Image-text matching** 判断 pair 是否匹配，并可通过 fusion encoder 建模更复杂交互。
- **Masked modeling** 根据上下文预测隐藏的 image patch、audio frame、video tubelet 或 text token。
- **Autoregressive generation** 在前文条件下预测下一个 token 或 modality output。
- **Distillation 与 pseudo-labeling** 把 specialist/teacher 的知识迁移到另一模态或统一 student。

联合训练通常组合多个 loss：

$$
\mathcal{L}=\lambda_{\text{contrast}}\mathcal{L}_{\text{contrast}}
+\lambda_{\text{match}}\mathcal{L}_{\text{match}}
+\lambda_{\text{mask}}\mathcal{L}_{\text{mask}}
+\lambda_{\text{gen}}\mathcal{L}_{\text{gen}}.
$$

各系数决定 gradient scale 与 representation priority。增加 objective 可能产生 interference，而不是免费获得能力。[BLIP](https://arxiv.org/abs/2201.12086)展示了如何在过滤 noisy caption 的同时组合 understanding 和 generation；[FLIP](https://arxiv.org/abs/2212.00794)则通过 image masking 提高 language-image pretraining efficiency。

下面的代码保持已训练 dual encoder 冻结，用显式 negative pair 学习 image-text matching head，并独立训练 masked-pixel reconstructor。它们是小型 objective probe，不是统一 production model。

<details>
<summary><strong>PyTorch：探测 matching 与 masked reconstruction objective</strong></summary>

```python
with torch.no_grad():
    image_features = dual_encoder.encode_image(train_x[:300])
    text_features = dual_encoder.encode_text(train_text_ids[:300], train_text_mask[:300])
labels_300 = train_y[:300]
negative_rows = []
for row, label in enumerate(labels_300):
    candidate = (row + 1) % len(labels_300)
    while labels_300[candidate] == label:
        candidate = (candidate + 1) % len(labels_300)
    negative_rows.append(candidate)
negative_rows = torch.tensor(negative_rows)

def pair_features(left, right):
    # Matching needs an interaction term; concatenated marginals alone cannot
    # express whether two vectors agree with a single linear decision layer.
    return torch.cat([left * right, (left - right).abs()], dim=1)


positive_pairs = pair_features(image_features, text_features)
negative_pairs = pair_features(image_features, text_features[negative_rows])
match_inputs = torch.cat([positive_pairs, negative_pairs], dim=0)
match_targets = torch.cat([torch.ones(300), torch.zeros(300)]).long()
match_train_rows = torch.cat([torch.arange(240), torch.arange(300, 540)])
match_eval_rows = torch.cat([torch.arange(240, 300), torch.arange(540, 600)])
matching_head = nn.Linear(64, 2)
optimizer = torch.optim.AdamW(matching_head.parameters(), lr=1e-2)
for _ in range(80):
    optimizer.zero_grad()
    matching_loss = F.cross_entropy(
        matching_head(match_inputs[match_train_rows]), match_targets[match_train_rows]
    )
    matching_loss.backward()
    optimizer.step()
matching_accuracy = float((
    matching_head(match_inputs[match_eval_rows]).argmax(1) == match_targets[match_eval_rows]
).float().mean())

seed_everything(1370)
mask = torch.rand_like(train_x[:300]) < 0.35
masked_images = train_x[:300].masked_fill(mask, 0.0)
reconstructor = nn.Sequential(nn.Flatten(), nn.Linear(64, 96), nn.ReLU(), nn.Linear(96, 64))
optimizer = torch.optim.AdamW(reconstructor.parameters(), lr=3e-3)
for _ in range(80):
    optimizer.zero_grad()
    reconstruction = reconstructor(masked_images).view_as(masked_images)
    masked_loss = F.mse_loss(reconstruction[mask], train_x[:300][mask])
    masked_loss.backward()
    optimizer.step()

assert matching_accuracy > 0.5
assert torch.isfinite(masked_loss)
print({"matching accuracy on held-out pairs": round(matching_accuracy, 3),
       "masked-pixel reconstruction MSE": round(float(masked_loss.detach()), 4)})
```

</details>

Negative construction 是 matching task 的一部分：简单 random negative 可能奖励表面不匹配，而 hard negative 可能意外成为有效 pair。Reconstruction quality 也可能优先建模 texture 而不是 semantics。因此 objective evaluation 应包括 transfer probe、retrieval、grounding、generation 与 ablation，不能假定 pretraining loss 覆盖所有期望能力。

### **适配、评估与能力迁移** {#adaptation-evaluation-capability-transfer}

基础模型评估是 capability、modality、task、domain 与 adaptation budget 构成的矩阵。至少应该区分：

- **Zero-shot transfer：**不更新目标参数。
- **Few-shot 或 linear probing：**在 frozen representation 上使用少量目标标签。
- **PEFT/full tuning：**逐步增加更新容量，对应第 12 章内容。
- **Retrieval 与 grounding：**匹配概念和局部证据是否对齐。
- **Robustness 与 modality ablation：**行为能否承受 shift、missing input 或 conflict。

对于 class-level retrieval，Recall@$k$ 检查有效类别文本是否出现在 top $k$ text score 中。Linear probe 检查 frozen representation 是否暴露目标边界。两者都不能证明 generative faithfulness。Prompted generation 还需要在指定 rubric 下评估 factuality、calibration、refusal behavior 与 human evaluation。

下面的审计使用共享 embedding 评估 zero-shot digit recognition、class retrieval、parity probe 与 shifted-image stress test。偏移与第 12 章使用的 label-preserving corruption family 一致。

<details>
<summary><strong>Python：评估 transfer、retrieval、shift 与 missing modality</strong></summary>

```python
def shifted_domain(images, seed=1380):
    shifted = torch.zeros_like(images)
    shifted[:, :, :, 1:] = images[:, :, :, :-1]
    blurred = F.avg_pool2d(shifted, 3, stride=1, padding=1)
    generator = torch.Generator().manual_seed(seed)
    noise = 0.05 * torch.randn(images.shape, generator=generator)
    return (0.84 * blurred + noise).clamp(0.0, 1.0)


dual_encoder.eval()
with torch.no_grad():
    train_embeddings = dual_encoder.encode_image(train_x).numpy()
    test_embeddings = dual_encoder.encode_image(test_x).numpy()
    test_image_embeddings = torch.tensor(test_embeddings)
    similarities = test_image_embeddings @ ensemble_prototypes.T
    ranking = similarities.argsort(dim=1, descending=True)
    recall_at_1 = float((ranking[:, :1] == test_y[:, None]).any(dim=1).float().mean())
    recall_at_5 = float((ranking[:, :5] == test_y[:, None]).any(dim=1).float().mean())
    shifted_accuracy = zero_shot_score(shifted_domain(test_x), test_y, ensemble_prototypes)

parity_probe = LogisticRegression(max_iter=500, random_state=1380)
parity_probe.fit(train_embeddings, (train_y.numpy() % 2))
parity_accuracy = accuracy_score(test_y.numpy() % 2, parity_probe.predict(test_embeddings))

intermediate_model = fusion_models["intermediate"]
intermediate_model.eval()
with torch.no_grad():
    missing_hint_accuracy = float((
        intermediate_model(test_x, torch.zeros_like(test_hints)).argmax(1) == test_y
    ).float().mean())

assert recall_at_5 >= recall_at_1
print({
    "zero-shot / retrieval R@1": round(recall_at_1, 3),
    "retrieval R@5": round(recall_at_5, 3),
    "frozen embedding parity probe": round(float(parity_accuracy), 3),
    "shifted zero-shot accuracy": round(shifted_accuracy, 3),
    "intermediate fusion with hint missing": round(missing_hint_accuracy, 3),
})
```

</details>

由于本实验文本只包含 class-level semantics，zero-shot classification 与 R@1 在数值上等价；真实 retrieval 包含多个 caption 与多个有效 image，因此 image-to-text 和 text-to-image recall 会不同。Capability transfer 应与 task-specific baseline 和 uncertainty interval 比较。如果开发期间反复在同一个 benchmark 上选择模型，那么该 benchmark 已经成为开发过程的一部分，需要新的 final test。

### **基础模型生命周期** {#foundation-model-lifecycle}

基础模型生命周期是一组版本化 artifact 与 gate：

1. **问题与政策定义：**intended use、excluded use、risk tolerance、data rights 与 success criteria。
2. **数据流水线：**provenance、mixture weight、filtering、deduplication、deletion lineage 与 held-out evaluation boundary。
3. **预训练：**architecture、objective、compute、checkpoint、scaling evidence 与 incident log。
4. **适配：**prompt、retrieval、PEFT、full tuning、tool 与 use-specific safety control。
5. **评估与发布：**capability、robustness、safety、privacy、security、subgroup 与 systems evidence。
6. **监控与修订：**drift、abuse、feedback quality、rollback、deprecation 与 model/data version link。

Model card 和数据文档不能替代评估，但可以让假设更容易被发现。发布决策应该明确哪些证据适用于 base，哪些只适用于某个 adapted system。API 更新、安全政策变化或新的 retrieval corpus，即使 model identifier 看起来没有变化，也可能改变行为。

<details>
<summary><strong>Python：建立可审计的评估记录与 drift signal</strong></summary>

```python
with torch.no_grad():
    clean_embeddings = dual_encoder.encode_image(test_x)
    shifted_embeddings = dual_encoder.encode_image(shifted_domain(test_x, seed=1390))
    paired_cosine = F.cosine_similarity(clean_embeddings, shifted_embeddings, dim=1)
    embedding_drift = float(1.0 - paired_cosine.mean())

evaluation_record = {
    "model": {"family": "tiny dual encoder", "version": "dl13-demo-v1"},
    "data": {
        "source": "UCI Optical Recognition of Handwritten Digits via scikit-learn",
        "doi": "10.24432/C50P49",
        "license": "CC BY 4.0",
        "constructed_text": True,
        "split_seed": 1313,
    },
    "evidence": {
        "prompt_ensemble_zero_shot": round(ensemble_score, 3),
        "shifted_zero_shot": round(shifted_accuracy, 3),
        "mean_embedding_drift": round(embedding_drift, 3),
        "missing_hint_accuracy": round(missing_hint_accuracy, 3),
    },
    "limitations": [
        "class-level synthetic captions",
        "ten classes and small images",
        "mechanism demonstration rather than a VLM benchmark",
    ],
}
assert evaluation_record["data"]["constructed_text"] is True
assert 0.0 <= embedding_drift <= 2.0
print(evaluation_record)
```

</details>

Production monitor 的 threshold 应根据正常变化拟合，而不是使用任意常数。Embedding drift 可以发现变化，却不能自动识别伤害；embedding 不变时，decoder behavior 或 policy 仍可能改变。应该共同监控 input distribution、retrieval source、output quality、safety event、latency 与 human escalation，并保留足够 lineage，以重现 incident 发生时准确启用的 base、adapter、prompt、index 与 policy。

### **从预训练到后训练** {#pretraining-to-post-training}

预训练从大规模 corpus 学习广泛统计结构。Continued pretraining 可以让 base 专门适应某个 domain 或 time period。Supervised/instruction tuning 教授目标 response format 与 task following。Preference optimization、reinforcement learning、critique、rejection sampling 与 safety tuning 再通过 comparison、reward 或 policy constraint 塑造行为。Serving 阶段还会加入 system prompt、retrieval、tool、filter 与 monitoring。

这些阶段解决不同问题。Pretraining loss 奖励预测 corpus regularity，而不是真实性或用户意图；instruction tuning 可以改善 interface compliance，却不一定增加可靠 world knowledge；preference alignment 可以改变 style，也可能牺牲部分 capability；retrieval 无需修改参数就能更新 evidence，但会引入 source quality 与 prompt-injection risk。

一个有用的依赖图为

```text
curated pretraining data
        -> base representations and capabilities
        -> continued pretraining or adaptation
        -> instruction/task tuning
        -> preference and safety alignment
        -> retrieval/tools/policies at serving time
        -> monitored deployment evidence
```

每个改变行为的阶段之后都应该重新评估。Base-model benchmark 无法认证最终 application，最终 application test 也不能揭示所有上游 capability 或 privacy risk。第 18 章会展开 supervised fine-tuning、reward modeling、PPO 和 DPO 等 post-training objective；本章只说明它们在生命周期中的位置。

### **章节比较与总结** {#chapter-comparison-summary}

基础模型围绕可复用上游资产和多个下游适配来组织深度学习。多模态模型把这些资产扩展到具有不同 sampling rate、token budget、noise process 和 semantics 的信号。因此，它的设计范围远大于选择一个 Transformer block。

| 设计问题 | 主要选择 | 必须测量的内容 |
|---|---|---|
| Scale allocation | parameter、data、step、compute | held-out loss、不确定性、推理成本 |
| Capacity | dense 或 sparse MoE | total/active parameter、FLOPs、routing 与 communication |
| Modality interface | 独立 encoder 或 unified token | 保留的信息、mask、token balance |
| Fusion | early、intermediate、late | complementarity、missing/conflicting modality |
| Alignment | shared embedding 或 token-level cross-attention | retrieval、grounding、pairwise compute |
| Objective | contrastive、matching、masked、generative | transfer、interference、shortcut behavior |
| Adaptation | prompting、retrieval、PEFT、full tuning | 目标质量、retention、memory、latency |
| Governance | 文档、release gate、monitoring | lineage、subgroup risk、incident、rollback |

本章的小型 image-text 实验揭示了几条普遍规律。Caption 与 negative sampling 共同定义 contrastive task；shared embedding 支持廉价 retrieval，却压缩局部证据；除非任务真正要求某模态，fusion 可能忽略它；sparse expert 把总容量与 active computation 分开，却引入 routing 与 communication failure mode；scaling curve 只在特定 regime 内成立，而 data curation 对结果的影响可能与 architecture 同样大。

实际标准应该是一条可审计链路：记录 data provenance 与 constructed modality；明确 token 与 compute budget；在受控证据下比较 dense、sparse、alignment 和 fusion 选择；评估 zero-shot、adapted、shifted 与 missing-modality behavior；并把每次部署连接到准确的 model、data、retrieval、prompt 与 policy version。

第 14 章将转向生成建模本身，解释 probability model、maximum likelihood、autoregressive factorization、exposure bias 与 decoding。这些机制是理解许多 foundation model 所暴露 generative interface 的前提，但一个模型不必具有生成能力，仍然可以充当可复用 foundation。